
## Text Summarization




This system provides a multi-modal approach to text summarization, utilizing **BART** and **FLAN-T5** models to transform complex information into various styles. By leveraging advanced NLP techniques like hierarchical processing and copy-detection, it ensures summaries are both concise and original.

***Key Applications***
*   **Executive Briefing:** Use *Formal* mode to convert technical papers or reports into academic abstracts.
*   **Content Simplification:** Use *Casual* mode to explain sophisticated concepts to students or non-experts.
*   **Quick Scanning:** Use *Bullet* mode for a sentence-by-sentence breakdown of news articles or long emails.
*   **Synthesized Overviews:** Use *Paragraph* mode to merge multiple ideas from a passage into a cohesive, restructured summary.


### 0. Environment Setup & Core Libraries

To power our multi-modal summarization system, we utilize the following key libraries:

*   **`transformers` (Hugging Face):** This is the core library providing access to pre-trained state-of-the-art models.
    *   `AutoModelForSeq2SeqLM`: A generic model class that will instantiate one of the sequence-to-sequence model classes (like BART or T5) of the library when created with the `from_pretrained()` method.
    *   `AutoTokenizer`: Responsible for converting raw text into a format (numerical tokens) that the models can process.
    *   `logging`: Used here to manage and suppress non-critical warnings during model loading to keep the notebook output clean.
*   **`re` (Regular Expressions):** A standard Python module used for advanced text pattern matching, which we employ for sentence splitting, paragraph detection, and cleaning metadata from model outputs.

In [1]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers.utils import logging as hf_logging
import re

### I. Model Roles & Architecture

This system uses a hybrid approach to ensure both factual accuracy and stylistic flexibility:

1.  **BART (Bidirectional and Auto-Regressive Transformers):**
    *   **Role:** Heavy Lifting & Extraction.
    *   **Use Case:** BART is specifically tuned for CNN/Daily Mail summarization. We use it to condense long passages into 'content seeds' or to handle hierarchical summarization for very long texts. It is excellent at identifying the most important facts within a large block of text.

2.  **FLAN-T5 (Fine-tuned Language Net):**
    *   **Role:** Stylistic Rewriting & Paraphrasing.
    *   **Use Case:** FLAN-T5 is a versatile instruction-tuned model. We use it to take the facts extracted by BART and transform them into specific styles (Casual, Formal, Bullet). It is responsible for the 'tone' of the summary and ensuring the output isn't just a copy-paste of the original sentences.

In [2]:
# Load the models
BART_MODEL = "facebook/bart-large-cnn"
T5_MODEL = "google/flan-t5-base"
BULLET = "\u2022"

# Hide noisy Transformers warnings without changing model configs or weights.
hf_logging.set_verbosity_error()

print("Loading BART...")
bart_tokenizer = AutoTokenizer.from_pretrained(BART_MODEL)
bart_model = AutoModelForSeq2SeqLM.from_pretrained(BART_MODEL)

print("Loading FLAN-T5...")
t5_tokenizer = AutoTokenizer.from_pretrained(T5_MODEL)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL)

Loading BART...


/Users/macbook/.pyenv/versions/3.11.9/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/Users/macbook/.pyenv/versions/3.11.9/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading FLAN-T5...



### II. Text Utilities

To prepare the text for the models and handle formatting, we use several utility functions. These help with counting words accurately, splitting text into logical paragraphs, and ensuring the summaries follow correct grammatical structures.


In [3]:
def word_count(text):
    # Returns the raw word count based on whitespace splitting.
    return len(text.split())


def clean_word_count(text):
    # Returns the word count after stripping punctuation
    # and non-alphanumeric characters.
    return len(get_words(text))

def get_paragraphs(text):
    # Splits a text into a list of paragraphs based on double newlines.
    paragraphs = [paragraph.strip() for paragraph
                  in re.split(r"\n\s*\n", text) if paragraph.strip()]

    if not paragraphs:
        return [text.strip()]

    return paragraphs

In [4]:
def split_into_sentences(text):
    # Normalize whitespace by replacing multiple spaces/newlines with a single space
    normalized = re.sub(r"\s+", " ", text.strip())

    if not normalized:
        return []

    # Use regex to find sequences of characters ending in punctuation (.!?)
    # followed by a space or end-of-string (positive lookahead)
    sentences = re.findall(r"[^.!?]+[.!?]+(?=\s|$)", normalized)

    # Extract any trailing text that doesn't end with a punctuation mark
    trailing = re.sub(r".*[.!?]+\s*", "", normalized)

    # If there is trailing text and it's not the entire string, add it as the last sentence
    if trailing and trailing != normalized:
        sentences.append(trailing)

    # Fallback: if no sentences were found (e.g., no punctuation), treat the whole text as one sentence
    if not sentences:
        return [normalized]

    # Return cleaned list of sentences, removing extra padding whitespace
    return [sentence.strip() for sentence in sentences if sentence.strip()]

In [5]:
def split_into_chunks(text, chunk_size=400):
    # Splits long text into chunks of a specific word count to fit model token limits
    words = text.split()

    return [" ".join(words[index:index + chunk_size]) for index
            in range(0, len(words), chunk_size)]

def ensure_sentence(text):
    # Cleans up whitespace and ensures the string ends with proper punctuation
    text = " ".join(text.strip().split())

    if not text:
        return text
    if text[-1] not in ".!?":
        text += "."

    return text

def lowercase_first(text):
    # Converts the first character of a string to lowercase (useful for appending to prefixes)
    if not text:
        return text
    return text[0].lower() + text[1:]

In [6]:
def strip_list_marker(text):
    # Removes common bullet points or numbering from the start of a generated string
    marker_chars = "-*" + BULLET + "0123456789. )\t"
    return text.lstrip(marker_chars).strip()


def force_one_sentence(text):
    # Takes multiple sentences and fuses them into one using semicolons
    text = strip_list_marker(" ".join(text.split()))
    sentences = split_into_sentences(text)

    if len(sentences) <= 1:
        return ensure_sentence(text)

    # Join sentences with semicolons and ensure the final result is a single punctuated sentence
    fused = "; ".join(sentence.rstrip(".!?") for sentence
                      in sentences)
    return ensure_sentence(fused)

In [7]:
def strip_summary_intro(text):
    # Removes redundant introductory phrases like "This passage talks about..." to avoid double-starts
    text = force_one_sentence(text).rstrip(".!?")
    intro_patterns = [
        r"^this\s+(passage|paragraph|text|article)\s+"
        r"(talks about(?: how)?|gives you an idea about|gives an idea about|"
        r"discusses(?: how)?|explains|describes|covers|is about)\s+",
        r"^the\s+(passage|paragraph|text|article)\s+"
        r"(talks about(?: how)?|gives you an idea about|gives an idea about|"
        r"discusses(?: how)?|explains|describes|covers|is about)\s+",
    ]

    for pattern in intro_patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE).strip()
    return text


def apply_summary_prefix(text, prefix):
    # Cleans the generated text and prepends the required style-specific opening phrase
    body = strip_summary_intro(text)

    if not body:
        return ensure_sentence(prefix)

    # Handle cases where both the prefix and body might start with "how"
    if prefix.lower().endswith(" how") and body.lower().startswith("how "):
        body = body[4:].strip()

    return ensure_sentence(f"{prefix} {lowercase_first(body)}")

In [8]:
def is_meta_summary(text):
    # Detects if the model generated a 'meta' response talking
    # *about* a summary instead of providing one
    lowered = text.lower()
    meta_phrases = [
        "a short summary",
        "brief summary",
        "main idea of this passage",
        "main idea of the passage",
        "main idea of this text",
        "what the passage is about",
        "what this passage is about",
        "what the text is about",
        "the passage talks about",
        "the text talks about",
        "this passage is about",
        "this text is about",
    ]

    return any(phrase in lowered for phrase in meta_phrases)


def format_with_newlines(text, word_limit=20):
    # Inserts newlines at a specific word interval to prevent very long lines in the display
    words = text.split()
    formatted_lines = []
    current_line_words = []
    for i, word in enumerate(words):
        current_line_words.append(word)
        if (i + 1) % word_limit == 0:
            formatted_lines.append(" ".join(current_line_words))
            current_line_words = []
    if current_line_words:
        formatted_lines.append(" ".join(current_line_words))
    return "\n".join(formatted_lines)

### III. Copy Detection

To ensure the summaries are not just exact copies of the source text, we implement an n-gram based copy detection system. This process works by:

1.  **Tokenization:** Breaking both the source and the generated candidate summaries into normalized word lists.
2.  **N-gram Extraction:** Creating sets of overlapping word sequences (typically 4 words long).
3.  **Ratio Calculation:** Determining what percentage of n-grams in the summary also appear in the source.
4.  **Selection & Retry:** If a summary exceeds a specific 'copy threshold', the system either selects a different candidate or triggers a 'strong paraphrase' retry prompt to force the model to use its own words.

In [9]:
def get_words(text):
    # Extracts all words and numbers from the text, converting to lowercase for normalization.
    return re.findall(r"[A-Za-z0-9']+", text.lower())


def get_ngrams(text, n=4):
    # Converts text into a set of unique sequences of 'n' consecutive words (n-grams).
    words = get_words(text)

    return {tuple(words[index:index + n]) for index
            in range(len(words) - n + 1)}


def copied_ngram_ratio(source, candidate, n=4):
    # Calculates the percentage of n-grams in the candidate that are also present in the source.
    candidate_ngrams = get_ngrams(candidate, n)

    if not candidate_ngrams:
        return 0

    source_ngrams = get_ngrams(source, n)
    # The intersection (&) finds n-grams common to both the source and candidate.
    copied_ngrams = candidate_ngrams & source_ngrams

    return len(copied_ngrams) / len(candidate_ngrams)


def choose_least_copied(source, candidates):
    # Filters valid candidates and selects the one with the lowest overlap ratio with the source.
    usable = [force_one_sentence(candidate) for candidate in candidates
              if candidate and candidate.strip()]

    if not usable:
        return ""

    # Returns the candidate with the minimum copied_ngram_ratio.
    return min(usable, key=lambda candidate:
               copied_ngram_ratio(source, candidate))

### IV. FLAN-T5 Generation

FLAN-T5 (Fine-tuned Language Net) represents the **generative heart** of this experiment. While BART excels at extraction, FLAN-T5 is responsible for the nuanced transformation of data into specific personas. This is one of the most critical stages because:

*   **Instruction Following:** Unlike traditional models, FLAN-T5 is fine-tuned on thousands of instructions, allowing it to interpret complex constraints like "Write like you are explaining it to a 12-year-old" or "Use precise academic language."
*   **Stylistic Flexibility:** It acts as a 'neural editor,' taking the raw 'content seeds' (facts) and restructuring them to meet specific word counts and tonal requirements.
*   **Iterative Refinement:** We use FLAN-T5 in a multi-pass process. If the first attempt is too similar to the source (detected by our Copy Detection system), we feed the output back to FLAN-T5 with a 'strong paraphrase' instruction to force higher originality.
*   **Controlled Output:** Through the `style_core_idea` function, we manage the model's creativity to ensure it balances factual density with the desired linguistic style.

In [10]:
def t5_generate(prompt, max_tokens=64, num_return_sequences=1,
                do_sample=True, num_beams=1):
    # Encodes the input prompt into tokens compatible with the T5 model architecture.
    inputs = t5_tokenizer(prompt, return_tensors="pt",
                          truncation=True, max_length=1024)

    # Generates text using a variety of sampling and penalty constraints:
    # - top_p & temperature: Control the balance between randomness and focus.
    # - no_repeat_ngram_size: Prevents the model from getting stuck in repetitive loops.
    # - repetition_penalty: Discourages the model from repeating words from the prompt or its own output.
    outputs = t5_model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        num_beams=num_beams,
        do_sample=do_sample,
        top_p=0.92,
        temperature=0.95,
        no_repeat_ngram_size=3,
        repetition_penalty=1.2,
        num_return_sequences=num_return_sequences
        )

    # Decodes the numerical tokens back into human-readable text, removing special tokens like </s>.
    decoded = [t5_tokenizer.decode(output, skip_special_tokens=True).strip()
    for output in outputs]

    # Return a single string if only one sequence was requested, otherwise return the list.
    if num_return_sequences == 1:
        return decoded[0]

    return decoded

In [11]:
def t5_paraphrase(prompt, source, max_tokens=64, copy_threshold=0.28):
    # Step 1: Generate a set of 4 potential candidates using the standard T5 generation pipeline.
    candidates = t5_generate(prompt, max_tokens=max_tokens,
                             num_return_sequences=4)

    # Step 2: Select the candidate that has the least overlap with the original source text.
    result = choose_least_copied(source, candidates)

    # Step 3: Check if the best candidate still exceeds the allowed copy threshold.
    if copied_ngram_ratio(source, result) <= copy_threshold:
        return result

    # Step 4 (Conditional Retry): If the threshold is failed, construct a more restrictive prompt.
    # This forces the model to move away from the source vocabulary and structure.
    retry_prompt = f"""
    The previous rewrite copied too much wording from the original.

    Rewrite the idea again with stronger paraphrasing.

    Strict rules:
    - Preserve the meaning
    - Use a new sentence structure
    - Use synonyms where possible
    - Do not copy any phrase of four or more words
    - Keep unavoidable technical terms only
    - Return one sentence only

    Original:
    {source}
    """

    # Step 5: Generate a second round of candidates using the stricter 'strong paraphrase' instructions.
    retry_candidates = t5_generate(retry_prompt, max_tokens=max_tokens,
                                   num_return_sequences=4)
    retry = choose_least_copied(source, retry_candidates)

    # Return the improved retry if available, otherwise fallback to the initial best result.
    if retry:
        return retry

    return result

In [12]:
def t5_deterministic(prompt, max_tokens=64):
    # Executes a non-random generation by disabling sampling and using beam search.
    # This ensures the model always returns the same (theoretically most probable) result for a given prompt.
    return t5_generate(prompt, max_tokens=max_tokens,
                       do_sample=False, num_beams=4)

In [13]:
def choose_styled_summary(source, candidates,
                          min_words, max_words, required_prefix):
    # This function implements a custom scoring algorithm to pick the best generation based on constraints.
    usable = []
    fallback = []

    for candidate in candidates:
        if not candidate or not candidate.strip():
            continue

        # Standardize formatting to a single sentence and apply the required style prefix.
        candidate = force_one_sentence(candidate)

        if required_prefix:
            candidate = apply_summary_prefix(candidate, required_prefix)

        fallback.append(candidate)

        # Priority filter: separate actual content from 'meta' responses (e.g., 'This is a summary...').
        if not is_meta_summary(candidate):
            usable.append(candidate)

    # If all candidates are meta-summaries, we have no choice but to use the fallback list.
    if not usable:
        usable = fallback

    if not usable:
        return ""

    # Define the ideal 'sweet spot' for length calculation.
    target = (min_words + max_words) / 2

    def score(candidate):
        words = clean_word_count(candidate)
        # Calculate penalties for length violations.
        if min_words <= words <= max_words:
            word_penalty = 0
        else:
            word_penalty = min(abs(words - min_words), abs(words - max_words))

        # The selection score ranks candidates based on four weighted factors:
        # 1. Meta-content avoidance (most important)
        # 2. Hard length constraint violations
        # 3. Proximity to the target word count
        # 4. Uniqueness relative to the source (lowest copy ratio wins ties)
        return (int(is_meta_summary(candidate)), word_penalty,
                abs(words - target), copied_ngram_ratio(source, candidate))

    return min(usable, key=score)

In [14]:
def extract_core_idea(text):
    # Stage 1: Factual Extraction. Use BART to distill the raw facts from the source.
    # Large texts are processed hierarchically to avoid token limits.
    if word_count(text) > 500:
        content_seed = hierarchical_bart(text)
    else:
        content_seed = bart_summarize(
            text,
            min_len=30,
            max_len=90
        )

    # Stage 2: Content Planning. Use FLAN-T5 to turn extractive facts into a dense 'factual plan'.
    # The prompt explicitly forbids summary-like language to prevent recursive 'meta' generation.
    prompt = f"""
    Create a factual content plan from the passage.

    Rules:
    - State what the passage says, not what a summary should do
    - Include the main topic and the most important supporting points
    - Mention the actual subject matter directly
    - Do not use phrases like "main idea", "this passage", "this text", or "summary"
    - Do not include minor examples
    - Use one clear sentence
    - Use 30 to 45 words

    Passage:
    {text}

    Content seed:
    {content_seed}
    """

    # Generate multiple plans and pick the one that captures facts without copying source syntax.
    candidates = t5_generate(prompt, max_tokens=70,
                             num_return_sequences=6)
    non_meta_candidates = [force_one_sentence(candidate) for candidate in candidates
                           if candidate and not is_meta_summary(candidate)]

    if non_meta_candidates:
        return choose_least_copied(text, non_meta_candidates)

    # Fallback logic if the LLM fails to provide a non-meta core idea.
    if content_seed and not is_meta_summary(content_seed):
        return force_one_sentence(content_seed)

    return force_one_sentence(text)

In [15]:
def style_core_idea(original_text, core_idea,
                    style_name, style_rules,
                    min_words, max_words, max_tokens,
                    required_prefix):

    # Phase 1: Initial Stylistic Generation.
    # Combines the 'core idea' facts with specific 'style rules' (e.g., Casual vs Formal).
    prompt = f"""
    Rewrite the core idea as a {style_name} summary.

    The casual and formal summaries must describe the same core content.
    The summary must begin exactly with: {required_prefix}

    Style rules:
    {style_rules}

    Length rules:
    - Use {min_words} to {max_words} words total, including the required opening
    - Return one sentence only

    Anti-copy rules:
    - Use wording that is very different from the original passage
    - Do not copy any phrase of four or more words
    - Keep only necessary technical terms

    Core idea:
    {core_idea}

    Original passage:
    {original_text}
    """

    candidates = t5_generate(prompt, max_tokens=max_tokens,
                             num_return_sequences=6)

    result = choose_styled_summary(original_text, candidates,
                                   min_words, max_words, required_prefix)

    # If the first pass meets length requirements, return it immediately.
    if min_words <= clean_word_count(result) <= max_words:
        return result

    # Phase 2: Refinement Retry.
    # Triggered if Phase 1 failed word counts or had high copy ratios.
    retry_prompt = f"""
    Rewrite this same core idea again as a {style_name} summary.

    Core idea:
    {core_idea}

    Strict requirements:
    - Begin exactly with: {required_prefix}
    - Use {min_words} to {max_words} words total, including the required opening
    - One sentence only
    {style_rules}
    - Use wording very different from the original passage
    - Do not copy any phrase of four or more words

    Original passage:
    {original_text}
    """

    retry_candidates = t5_generate(retry_prompt,
                                   max_tokens=max_tokens,
                                   num_return_sequences=6)

    retry = choose_styled_summary(original_text, retry_candidates,
                                  min_words, max_words, required_prefix)

    if retry and min_words <= clean_word_count(retry) <= max_words:
        return retry

    # Phase 3: Final Repair.
    # Explicitly instructs the model to 'shorten' or 'expand' based on the specific deviation.
    best_result = retry or result
    current_words = clean_word_count(best_result)

    repair_prompt = f"""
    Fix the length of this {style_name} summary while keeping the same meaning.

    Current summary:
    {best_result}

    Strict requirements:
    - Begin exactly with: {required_prefix}
    - Use {min_words} to {max_words} words total
    - One sentence only
    - Keep the same core idea and key contents
    {style_rules}

    Core idea:
    {core_idea}
    """

    if current_words < min_words:
        repair_prompt += "\n- Add useful key content without adding minor details\n"
    elif current_words > max_words:
        repair_prompt += "\n- Shorten the sentence without losing the main idea\n"

    repair_candidates = t5_generate(repair_prompt, max_tokens=max_tokens,
                                    num_return_sequences=6)

    repair = choose_styled_summary(original_text, repair_candidates,
                                   min_words, max_words, required_prefix)

    # Return the repaired version if it passes filters, else fallback to the original best attempt.
    if (repair and not is_meta_summary(repair)
    and min_words <= clean_word_count(repair) <= max_words):
        return repair

    return apply_summary_prefix(best_result, required_prefix)

### V. BART Summarization

BART (Bidirectional and Auto-Regressive Transformers) serves as the **extractive engine** of our pipeline. While FLAN-T5 is used for stylistic rewriting, BART is optimized for identifying and condensing the most salient factual information from raw text.

***Architectural Strengths***

*   **Denoising Autoencoder:** BART is trained by corrupting text with an arbitrary noising function and learning to reconstruct the original text. This makes it particularly robust at 'cleaning' and summarizing noisy real-world data.
*   **Encoder-Decoder Synergy:** It combines a bidirectional encoder (like BERT) with an autoregressive decoder (like GPT), allowing it to excel at sequence-to-sequence tasks where understanding the full context is vital before generating a summary.

***Implementation Strategy***

In this system, BART is utilized in two primary ways:
1.  **Content Seeding:** It generates the initial factual 'seed' that informs the T5 stylistic passes, ensuring that even if T5 is being creative with tone, it remains anchored to the source facts.
2.  **Hierarchical Summarization:** To bypass the 1024-token limit of standard Transformer models, we implement a recursive chunking strategy. Long documents are split into segments, summarized individually by BART, and then merged and re-summarized to maintain a coherent narrative flow without losing key details from any section.

In [16]:
def bart_summarize(text, min_len=20, max_len=100):
    # Tokenize the input text for BART, truncating if it exceeds the 1024 token limit
    inputs = bart_tokenizer(text, return_tensors="pt",
                            truncation=True, max_length=1024)

    # Generate the summary using beam search (num_beams=6)
    # length_penalty > 1.0 encourages the model to generate longer sequences
    # no_repeat_ngram_size=3 prevents the model from repeating 3-word phrases
    summary_ids = bart_model.generate(
        inputs["input_ids"],
        num_beams=6,
        min_length=min_len,
        max_length=max_len,
        no_repeat_ngram_size=3,
        length_penalty=2.0,
        early_stopping=True
        )

    # Decode the generated tokens back into a clean string
    return bart_tokenizer.decode(summary_ids[0],
                                 skip_special_tokens=True).strip()


def hierarchical_bart(text):
    # For texts longer than the model's context window, we split them into 400-word chunks
    chunks = split_into_chunks(text, chunk_size=400)
    partial_summaries = []

    # Summarize each chunk individually to capture local details
    for chunk in chunks:
        partial_summaries.append(bart_summarize(chunk,
                                                min_len=20,
                                                max_len=80))

    # Merge the individual summaries into one text block
    merged = " ".join(partial_summaries)

    # Perform a final 'master' summary on the merged text to produce a cohesive overview
    return bart_summarize(merged, min_len=30, max_len=120)

## VI. Specialized Summary Modes#

This section defines the core logic for our four distinct summarization personas. Each mode utilizes a specific combination of prompts, word-count constraints, and stylistic rules to transform the 'Core Idea' into a tailored output.

***1. Casual Mode***
*   **Goal:** Simplicity and accessibility.
*   **Target:** 20-25 words.
*   **Transformation:** Uses a 'Compression Operator' that simplifies vocabulary. It is designed to explain complex topics as if speaking to a student, using friendly, common verbs like *shows* or *helps*.

***2. Formal Mode***
*   **Goal:** Academic elevation and precision.
*   **Target:** 45-55 words.
*   **Transformation:** This mode expands on the core facts by incorporating broader implications and scientific terminology. It favors sophisticated verbs like *facilitates*, *demonstrates*, or *examines*.

***3. Bullet Mode***
*   **Goal:** Granular breakdown and scanning.
*   **Transformation:** Instead of summarizing the whole text at once, this mode iterates through every sentence in every paragraph. Each sentence is independently paraphrased and condensed into a single bullet point, ensuring no information is lost while improving readability.

***4. Paragraph Mode***
*   **Goal:** Cohesion and restructuring.
*   **Transformation:** This mode focuses on 'Sentence Fusion'. It takes groups of sentences and attempts to merge them into a single, fluid summary sentence per paragraph. For very long paragraphs, it triggers the Hierarchical BART logic to ensure the fusion remains factually dense.

In [17]:
def casual_summary(text):
    # Obtain the factual 'core idea' to ensure accuracy before applying style.
    core_idea = extract_core_idea(text)

    # The 'Compression Operator' for casual mode: forces simple synonyms and
    # uses a '12-year-old' persona to explain complex topics concisely.
    style_rules = """
    - Sound casual and easy to understand
    - Write like you are explaining it to a 12-year-old student
    - Give the main idea and key contents, not tiny details
    - Prefer short, common words
    - Use simple synonyms for words from the passage
    - Use friendly wording such as "shows", "helps", "learns", or "uses" when suitable
    - Avoid technical wording unless it is necessary
    """

    # Generate and refine the summary based on the 20-25 word target.
    result = style_core_idea(
        original_text=text,
        core_idea=core_idea,
        style_name="casual",
        style_rules=style_rules,
        min_words=20,
        max_words=25,
        max_tokens=45,
        required_prefix="This passage talks about how"
        )

    return format_with_newlines(ensure_sentence(result))

In [18]:
def formal_summary(text):
    # Extract core facts using the hybrid BART/T5 extraction pipeline.
    core_idea = extract_core_idea(text)

    # The 'Academic Elevation' operator: uses precise, scientific vocabulary
    # and expands on the significance of the facts rather than just the facts themselves.
    style_rules = """
    - Use formal, academic, and scientific language
    - Use more precise and elevated vocabulary than the casual version
    - Include the main idea and key contents
    - Expand the explanation with cause, function, significance, and broader implication
    - Prefer words such as "examines", "demonstrates", "facilitates", "empirical", "computational",
    or "systematic" when suitable
    - Avoid informal wording
    - Keep the same central content as the casual summary
    """

    # Generate the formal abstract with a target of 45-55 words.
    result = style_core_idea(
        original_text=text,
        core_idea=core_idea,
        style_name="formal academic",
        style_rules=style_rules,
        min_words=45,
        max_words=55,
        max_tokens=95,
        required_prefix="This passage discusses how"
        )

    return format_with_newlines(ensure_sentence(result))

In [19]:
def compress_sentence_to_bullet(sentence):
    # A focused prompt that forces the model to distill a single sentence into a bullet.
    # Uses a high copy_threshold check to ensure the bullet isn't just a copied fragment.
    prompt = f"""
    Rewrite this sentence as one concise bullet point using heavy paraphrasing.

    Strict rules:
    - Keep the original meaning
    - Use fresh wording
    - Change the sentence structure
    - Do not copy any phrase of four or more words
    - Keep unavoidable technical terms only
    - Remove unnecessary wording
    - Return one sentence only

    Sentence:
    {sentence}
    """

    result = t5_paraphrase(prompt, sentence,
                           max_tokens=48, copy_threshold=0.22)

    if not result:
        result = sentence

    return f"{BULLET} {force_one_sentence(result)}"


def bullet_summary(text):
    # The 'Granular' mode: iterates through every logical sentence in the document.
    # This ensures that even minor details from different paragraphs are captured.
    bullets = []

    for paragraph in get_paragraphs(text):
        for sentence in split_into_sentences(paragraph):
            bullets.append(compress_sentence_to_bullet(sentence))

    return "\n".join(bullets)

In [20]:
def fuse_sentences_to_one(sentence_group):
    # The 'Sentence Fusion' logic: takes a block of text and instructs T5
    # to synthesize all mentioned ideas into a single, complex summary sentence.
    prompt = f"""
    Fuse these sentences into one heavily paraphrased summary sentence.

    Strict rules:
    - One sentence only
    - Restructure the ideas instead of listing them
    - Preserve the main meaning
    - Use fresh wording
    - Change the sentence structure
    - Do not copy any phrase of four or more words
    - Keep unavoidable technical terms only
    - Do not use bullet points
    - Maximum 45 words

    Sentences:
    {sentence_group}
    """

    result = t5_paraphrase(prompt, sentence_group,
                           max_tokens=70, copy_threshold=0.22)

    if not result:
        return ensure_sentence(sentence_group)

    return force_one_sentence(result)


def paragraph_summary(text):
    # The 'Synthesized Overview' mode: processes each paragraph and merges its facts.
    # For very long paragraphs, it uses Hierarchical BART first to reduce noise.
    fused_paragraphs = []

    for paragraph in get_paragraphs(text):
        if word_count(paragraph) > 500:
            extractive_summary = hierarchical_bart(paragraph)
            fused = fuse_sentences_to_one(extractive_summary)
        else:
            sentences = split_into_sentences(paragraph)
            fused = fuse_sentences_to_one(" ".join(sentences))

        fused_paragraphs.append(ensure_sentence(fused))

    if len(fused_paragraphs) == 1:
        return format_with_newlines(fused_paragraphs[0])

    return format_with_newlines(" ".join(fused_paragraphs))

### VII. Summary Orchestration & Execution

The final stage of our pipeline is the **Summarization Router**. This function acts as a central dispatcher, receiving the raw text and a mode selection from the user.

***The Routing Logic***
*   **Switching Mechanism:** Depending on the `mode` parameter, the router triggers the specific algorithmic path (e.g., calling `casual_summary` for simplified output or `bullet_summary` for granular breakdown).
*   **Standardization:** The router ensures that all inputs are normalized (lowercased) and provides error handling for unsupported modes.
*   **End-to-End Testing:** The demonstration block below showcases the system's ability to process a complex, multi-paragraph passage about Machine Learning and transform it into four distinct linguistic styles simultaneously.

In [21]:
# Main Function: The Summarization Router
def summarize_text(text, mode="formal"):
    """
    This central dispatcher handles mode selection, routing the raw text
    to the specific algorithmic pipeline requested by the user.
    """
    mode = mode.lower()

    if mode == "casual":
        return casual_summary(text)

    if mode == "formal":
        return formal_summary(text)

    if mode == "bullet":
        return bullet_summary(text)

    if mode == "paragraph":
        return paragraph_summary(text)

    # Raises an error if an unsupported mode is passed to prevent silent failures.
    raise ValueError("mode must be one of: casual, formal, bullet, paragraph")

In [23]:
# Multi-paragraph sample text for the demonstration run.
sample_text = """
The rapid evolution of machine learning (ML) has fundamentally altered how humanity interacts with technology, acting as the primary engine driving the broader field of artificial intelligence.

At its core, machine learning represents a profound paradigm shift in computer science, moving away from rigid, human-authored rules toward dynamic systems that derive their own logic directly from empirical evidence.

Machine learning eradicates this bottleneck by utilizing statistical models and computational algorithms to learn the underlying rules of a domain through exposure to vast amounts of data.

Generalization is the ultimate goal of machine learning; it ensures that algorithms do not merely memorize training examples but extract meaningful patterns that allow them to make accurate predictions when exposed to new data.
"""

# Iterates through all available modes to show stylistic differences.
for mode in ["casual", "formal", "bullet", "paragraph"]:
    print("\n" + "=" * 60)
    print(f"MODE: {mode.upper()}")
    print("=" * 60)
    print(summarize_text(sample_text, mode))


MODE: CASUAL
This passage talks about how machine learning is a software that allows humans to understand and apply computer algorithms and
data.

MODE: FORMAL
This passage discusses how machine learning is a new form of generative computer science that has dramatically changed the way
we interact with technology; It has become increasingly computationally driven by applications to learning that derive their own logic directly
from empirical evidence.

MODE: BULLET
• ML has radically changed how humans interact with technology, and enables human life to be fully automated.
• Its foundation is in computer science, and it is moving away from human-authored rules and toward dynamic systems; They derive their logic from empirical evidence.
• Machine learning helps to fix this bottleneck and also improves the performance of the system through its ability to analyze large amounts of data.
• The goal of machine learning is to make it able to make predictions that are accurate, but that do no

#### VII. Baseline Approach: The Lead-3 Method

Before concluding, it is important to consider the **Baseline**. In the field of Natural Language Processing (NLP), a baseline is a simple, heuristic-based approach used to measure the relative improvement provided by complex neural models.

**The Lead-3 Practice:**
This method involves simply taking the first three sentences of a document and presenting them as the summary.

*   **Why it works:** In news articles and many formal documents, the most important information is often 'front-loaded' in the lead paragraph (the inverted pyramid style of journalism).
*   **The Limitation:** While surprisingly effective for news, this practice is simple and lacks the ability to synthesize information, handle varied styles, or condense long-tail details found later in the text.

In [24]:
def baseline_lead_n(text, n=3):
    # Split the entire text into logical sentences
    all_sentences = []
    for paragraph in get_paragraphs(text):
        all_sentences.extend(split_into_sentences(paragraph))

    # Select the first n sentences
    baseline = " ".join(all_sentences[:n])
    return format_with_newlines(baseline)

# Demonstration of the baseline
print("=" * 60)
print("BASELINE SUMMARY (Lead-3)")
print("=" * 60)
print(baseline_lead_n(sample_text, n=3))

BASELINE SUMMARY (Lead-3)
The rapid evolution of machine learning (ML) has fundamentally altered how humanity interacts with technology, acting as the primary engine
driving the broader field of artificial intelligence. At its core, machine learning represents a profound paradigm shift in computer science,
moving away from rigid, human-authored rules toward dynamic systems that derive their own logic directly from empirical evidence. Machine learning
eradicates this bottleneck by utilizing statistical models and computational algorithms to learn the underlying rules of a domain through exposure
to vast amounts of data.


### Another Example:

In [25]:
fast_text = """
Fast food has become a ubiquitous part of modern life, offering an unbeatable combination of convenience and affordability for busy schedules. While these readily available meals—from industry giants like McDonald's to local eateries—solve immediate time and financial constraints, they also present significant health challenges due to their high calorie and fat content.

The most prominent advantage of the fast-food industry is its sheer convenience and economic efficiency. In an increasingly fast-paced world, individuals often lack the time or energy to prepare home-cooked meals after a long day of work or school. Fast food provides an immediate, budget-friendly solution to feed hunger on the go. Furthermore, its global expansion has created extensive supply chains, contributing significantly to job creation and local economic growth. For millions of people, it is not just a treat, but a practical daily necessity.

Despite these practical benefits, the frequent consumption of fast food carries severe nutritional and medical drawbacks. Meals from establishments like Burger King or Domino's are typically heavily processed and packed with excessive amounts of saturated fats, refined sugars, and sodium. Over time, this nutritional profile is heavily linked to a rise in diet-related chronic conditions, including obesity, type 2 diabetes, and cardiovascular disease. Relying on these foods as a dietary staple can ultimately place a massive burden on personal well-being and healthcare systems.

In conclusion, fast food functions as a double-edged sword in contemporary society. While it successfully serves the short-term needs of affordability and rapid service, its long-term health consequences cannot be ignored. Striking a balance is crucial; treating fast food as an occasional, convenient indulgence rather than a primary diet allows individuals to enjoy its benefits while safeguarding their overall health
"""

for mode in ["casual", "formal", "bullet", "paragraph"]:
    print("\n" + "=" * 60)
    print(mode.upper())
    print("=" * 60)
    print(summarize_text(fast_text, mode))


CASUAL
This passage talks about how the calorie and high-calorie content of fast food can cause many health problems, including obesity
and diabetes.

FORMAL
This passage discusses how fast food functions as a double-edged sword in contemporary society; While it successfully serves short-term needs
of affordability and rapid service, its long-term health consequences cannot be ignored.

BULLET
• Fast food is the mainstay of modern life, offering a pricey, convenient and affordable option to busy people.
• Despite being readily available, the meals are also high in calorie and fat content.
• Fast-food is an excellent product; It makes a fast-food meal very easy.
• There are more and more people out there who cannot cook meals after work or school because they are not having enough time or energy.
• Fast food is a fast-food item to make meals affordable.
• Moreover, its global expansion has led to a large supply chain, which has significantly contributed to job creation and local econom

In [26]:
# Demonstration of the baseline
print("=" * 60)
print("BASELINE SUMMARY (Lead-3)")
print("=" * 60)
print(baseline_lead_n(fast_text, n=3))

BASELINE SUMMARY (Lead-3)
Fast food has become a ubiquitous part of modern life, offering an unbeatable combination of convenience and affordability for busy
schedules. While these readily available meals—from industry giants like McDonald's to local eateries—solve immediate time and financial constraints, they also
present significant health challenges due to their high calorie and fat content. The most prominent advantage of the fast-food industry
is its sheer convenience and economic efficiency.


## Recap

This notebook demonstrated a sophisticated **multi-modal summarization pipeline** that moves beyond simple text-shortening to true stylistic transformation.

***System Summary***
1.  **Extraction (BART):** We utilized BART-Large-CNN to perform the initial 'heavy lifting,' distilling raw, long-form text into dense factual seeds. This stage ensures the system remains factually grounded.
2.  **Transformation (FLAN-T5):** We leveraged the instruction-tuned capabilities of FLAN-T5 to act as a neural editor, applying specific personas (Casual vs. Formal) and structural constraints (Bullet vs. Paragraph).
3.  **Quality Control:** By integrating an **n-gram copy-detection system**, the pipeline proactively identifies and fixes instances where the model relies too heavily on source text, triggering automatic 'strong paraphrase' retries.

***Key Takeaways***
*   **Versatility:** The same underlying factual data can be successfully reshaped into a 20-word casual explanation or a 50-word academic abstract.
*   **Granularity:** Through 'Sentence Decomposition' in Bullet mode and 'Sentence Fusion' in Paragraph mode, the system offers users control over how information density is presented.
*   **Scalability:** The hierarchical processing logic allows the system to summarize documents of nearly any length while respecting transformer token limits.

This modular architecture serves as a robust template for building production-ready NLP tools that require both high precision and tonal flexibility.